# 08 · Give the agent a playbook

**Name + description → discovery**

**Instruction body → how to do the task**

A skill is a Markdown file. We will create two real skills for the teaching loop in notebook 09.

**Who does what:** Nano coordinates the skill-writing tools. Python writes and loads the lesson templates. This step needs no Lightning call.

###  instruction skills

[OpenClaw skills](https://docs.openclaw.ai/tools/skills) for discovery and loading, and the [NVIDIA skill catalog](https://github.com/NVIDIA/skills) for upstream examples. The terminal image pins OpenClaw 2026.9.7; the website describes the current release.

write the two prepared skills, inspect their frontmatter/body, and discover them. Manual file authoring requires no model service. Nano/OpenShell are needed only for the later agent handoff.

**Two consumers:** the `student-skills` files are read by our Python teaching loop in 09. OpenClaw separately reads the workshop orchestration skills installed into its sandbox workspace. A hash identifies the loaded text; it is not proof of behavioral quality.

**Try a focused experiment:** after 09 loads its tools, edit the copied `draft-report/SKILL.md` in that run's `lesson.work / 'student-skills'` to request a shorter summary, before running the loop. Compare with a fresh baseline run. Keep the skill name unchanged: Python also enforces action order by skill name, so prose cannot override those constraints. Inspect [the authoring source](agent/lessons/08-agent-skills.json).


### Start here
Run the short cells in order. Helpers execute the existing lesson code and keep detailed logs expandable. All displayed results come from your run. Starting again creates a fresh output folder.

In [ ]:
from pathlib import Path
import sys, json
from IPython.display import JSON, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course/common.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.agent.lesson_walkthrough import LessonWalkthrough, table, bars, loss_chart, show_skill_library
lesson = LessonWalkthrough('08-agent-skills')

### 1 · Write two small skills

| Skill | Instructions |
|---|---|
| draft-report | Observe → classify → check → draft or request review |
| describe-observation | Observe → finish |

These are fixed workshop templates, not model-generated instructions. The files live in this manual run’s `student-skills/` folder.

In [ ]:
lesson.step("write")
from IPython.display import Markdown
for name in ("draft-report", "describe-observation"):
    display(Markdown("#### " + name))
    print((lesson.work / "student-skills" / name / "SKILL.md").read_text())

### 2 · Discover and load the skill

The folder name and declared name must match. The description helps discovery; the loaded body tells the agent what to do. A hash identifies the exact version.

In [ ]:
lesson.step("discover")
table(["Skill", "Description"], [(name, item["description"]) for name,item in lesson.ns["skills"].items()])
print("Loaded instructions:\n", lesson.ns["body"])
print("Version hash:", lesson.ns["hashlib"].sha256(lesson.ns["loaded"].encode()).hexdigest())

### Check and save this manual run
Validate the artifacts and record this run for the next manual walkthrough. A successful check describes execution and evidence; read the actual quality scores separately.

In [ ]:
manual_result = lesson.finish()
display(JSON(manual_result, expanded=False))

### Grow the skill library
The library now includes this lesson’s skill, **workshop-write-skills**. Open the actual instructions below.

**`inspect-skills` → `write-skills` → `check-skills`**

In [ ]:
show_skill_library('08-agent-skills')

### Prepare OpenClaw
Run once for each new demo. This cell installs the growing skill library in a fresh sandbox workspace and starts a background host worker for this lesson’s three actions. **It does not ask the agent to do anything yet.**

Here, **installing a skill means placing its `SKILL.md` instruction file in OpenClaw’s workspace**. The current lesson’s source is `course/agent/skills/workshop-write-skills/SKILL.md` in this repository.

Each run gets this structure inside the OpenShell sandbox (launcher files omitted):

```text
/home/sandbox/course-runs/openclaw-<run-id>/
├── openclaw.json                 # points OpenClaw at this run’s workspace
├── chapter_tool.py              # helper invoked by the agent
├── chapter.json                 # this lesson’s task and allowed actions
└── workspace/
    ├── AGENTS.md                 # workspace instructions
    └── skills/
        ├── workshop-write-skills/
        │   └── SKILL.md
        └── ...                  # skills from earlier notebooks
```

OpenClaw discovers skills under `workspace/skills/`. The task you paste in the next step tells the agent which skill to read and supplies the full path to `chapter_tool.py` (the skill’s `<tool_path>` placeholder).

The agent invokes the helper through its execution tool. The helper forwards the selected action to the host worker, which runs the fixed lesson code and returns the result. Nano follows the skill and coordinates the calls.

This lesson does not require artifacts from an earlier OpenClaw demo.

Start the TUI task within **15 minutes** and keep this run’s `terminal_run` value for collecting results. If the worker expires, prepare a new run. The local run directory stores host results; the installed skills live at the sandbox paths printed below.


In [ ]:
from course.agent.chapter_terminal import prepare_chapter_terminal, collect_chapter_terminal
from course.agent.notebook_terminal import open_terminal
terminal_run = prepare_chapter_terminal('08-agent-skills')
print("Local run directory:", terminal_run["directory"])
print("OpenClaw sandbox workspace:", terminal_run["remote"] + "/workspace")
print("Installed skill:", terminal_run["remote"] + "/workspace/skills/workshop-write-skills/SKILL.md")
print("Python helper:", terminal_run["remote"] + "/chapter_tool.py")
print("\nThe agent will run these commands individually:")
for action in ['inspect-skills', 'write-skills', 'check-skills']:
    print("python3 " + terminal_run["remote"] + "/chapter_tool.py " + action)

### Open the TUI and paste the task
Exit the previous TUI with `/exit` first. This cell opens a terminal on the workshop computer’s desktop. Copy only the generated task into OpenClaw.

The terminal label is **Nano**, its lightweight orchestration model. Nano coordinates the skill-writing tools. Python writes and loads the lesson templates. This step needs no Lightning call.

In [ ]:
open_terminal(terminal_run)
print("\nPASTE THIS INTO OPENCLAW:\n")
print(terminal_run["prompt"])

### Bring the evidence back
After the TUI finishes, run this cell. It retrieves that session’s trace and checks the artifacts and skill reads. If Run All reaches this cell before you finish the TUI task, run it again afterward.

**Look for:** Two skill files created; names, instructions, and version hashes checked.

In [ ]:
agent_result = collect_chapter_terminal(terminal_run)
display(JSON(agent_result, expanded=True))

**What you demonstrated:** Our task skills define the workflow. Notebook 09 also loads NVIDIA’s pinned nemo-relay-instrument-calls skill to guide tracing. The teaching-loop skills are separate from OpenClaw’s growing workshop skill library.

Exit the TUI with `/exit` when done.